# Exercise 1 — validate_spec

Before building anything, validate the plan. `validate_spec` checks that a `CapstoneSpec` is complete enough to ship: all text fields are non-empty, there are at least 3 deliverables, at least 2 tech stack items, and at least 1 course section listed. It returns a `(bool, list)` tuple — the same pattern as Python's `issubset` and validation libraries.

In [ ]:
import datetime
from dataclasses import dataclass, field

@dataclass
class CapstoneSpec:
    name: str; tagline: str; domain: str; description: str
    sections_used: list; deliverables: list; tech_stack: list

@dataclass
class Phase:
    name: str; tasks: list; done: bool = False

@dataclass
class CapstoneReport:
    spec: CapstoneSpec
    phases: list = field(default_factory=list)
    started_at: str = field(default_factory=lambda: datetime.date.today().isoformat())
    completed_at: str = ""

_SPEC = CapstoneSpec(
    name          = "AI Trading Bot",
    tagline       = "Paper-trading bot with sentiment and technical signals.",
    domain        = "finance",
    description   = "End-to-end AI trading bot that fetches OHLCV data, computes "
                    "technical indicators, scores news sentiment with an LLM, applies "
                    "risk controls, and runs a daily paper-trading loop with logging.",
    sections_used = [
        "Section 3: Data & Analysis (pandas, SQLite)",
        "Section 4: Real Apps (FastAPI endpoint)",
        "Section 6: AI Agents (scheduling loop)",
        "Section 7: Finance & Trading (backtester, risk manager, paper trader)",
    ],
    deliverables  = [
        "paper_trader.py with buy/sell/portfolio_value",
        "bot_runner.py with daily scheduling and logging",
        "risk.py with stop-loss and drawdown controls",
        "Deployed FastAPI endpoint",
        "Portfolio case study",
    ],
    tech_stack    = ["Python", "pandas", "Ollama", "SQLite", "FastAPI"],
)

def validate_spec(spec):
    """Validate a CapstoneSpec for completeness.

    Returns:
        tuple[bool, list[str]]
          - (True, [])       if all rules pass
          - (False, errors)  if one or more rules fail

    Rules:
      - name, tagline, domain, description: non-empty string
      - deliverables: ≥ 3 items
      - tech_stack: ≥ 2 items
      - sections_used: ≥ 1 item
    """
    errors = []
    # Check text fields
    for field_name in ("name", "tagline", "domain", "description"):
        val = getattr(spec, field_name, "")
        if not isinstance(val, str) or not val.strip():
            errors.append(f"{field_name} must be a non-empty string")
    # TODO: check deliverables (≥ 3), tech_stack (≥ 2), sections_used (≥ 1)
    return len(errors) == 0, errors


### Checks

In [ ]:
checks = 0

# 1 — valid spec returns (True, [])
try:
    ok, errors = validate_spec(_SPEC)
    assert ok is True,  f"expected True, got {ok}"
    assert errors == [], f"expected [], got {errors}"
    checks += 1; print("✅ 1 valid spec → (True, [])")
except Exception as e:
    print("❌ 1:", e)

# 2 — empty name → error mentioning 'name'
try:
    bad = CapstoneSpec("","T","D","Desc",["S1"],["d1","d2","d3"],["py","pd"])
    ok, errors = validate_spec(bad)
    assert ok is False
    assert any("name" in err for err in errors), f"expected 'name' error, got {errors}"
    checks += 1; print("✅ 2 empty name → False with 'name' error")
except Exception as e:
    print("❌ 2:", e)

# 3 — fewer than 3 deliverables → error
try:
    bad = CapstoneSpec("N","T","D","Desc",["S1"],["only_one"],["py","pd"])
    ok, errors = validate_spec(bad)
    assert ok is False
    assert any("deliverable" in err for err in errors)
    checks += 1; print("✅ 3 < 3 deliverables → False with deliverables error")
except Exception as e:
    print("❌ 3:", e)

# 4 — fewer than 2 tech stack items → error
try:
    bad = CapstoneSpec("N","T","D","Desc",["S1"],["d1","d2","d3"],["py"])
    ok, errors = validate_spec(bad)
    assert ok is False
    assert any("tech" in err for err in errors)
    checks += 1; print("✅ 4 < 2 tech items → False with tech_stack error")
except Exception as e:
    print("❌ 4:", e)

# 5 — empty sections_used → error; multiple errors accumulate
try:
    bad = CapstoneSpec("N","T","D","Desc",[],["d1","d2","d3"],["py","pd"])
    ok, errors = validate_spec(bad)
    assert ok is False
    assert any("section" in err for err in errors)
    # 0 deliverables + 1 tech item would accumulate multiple errors
    worst = CapstoneSpec("","","","",[], [], [])
    ok2, errors2 = validate_spec(worst)
    assert len(errors2) >= 4, f"expected ≥4 errors for empty spec, got {len(errors2)}"
    checks += 1; print(f"✅ 5 empty sections_used → error; worst-case gives ≥4 errors")
except Exception as e:
    print("❌ 5:", e)

print(f"\n{checks}/5 checks passed!")
